In [1]:
import pandas as pd
import time 

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)


In [4]:
print("downloading fashion MNIST dataset(this may take 30-60 seconds)...")
fashion_mnist=fetch_openml('Fashion-MNIST',version=1,as_frame=False)
x=fashion_mnist.data
y=fashion_mnist.target.astype(int)
print(f"total dataset size:{x.shape[0]} images,each with {x.shape[1]} pixels.")

downloading fashion MNIST dataset(this may take 30-60 seconds)...
total dataset size:70000 images,each with 784 pixels.


In [5]:
x_subset,_,y_subset,_=train_test_split(
    x,
    y,
    train_size=12000,
    stratify=y,
    random_state=42
)



In [6]:
x_train,x_test,y_train,y_test=train_test_split(
    x_subset,
    y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)

In [7]:
print(f"Training image : {x_train.shape[0]}")
print(f"Testing image: {x_test.shape[0]}")


Training image : 10000
Testing image: 2000


In [8]:

print(f"before scaling -> Min: {x_train.min()},Max: {x_train.max()}")
x_train=x_train/255.0
x_test=x_test/255.0

before scaling -> Min: 0,Max: 255


In [9]:
print(f"after scaling  -> Min: {x_train.min()},Max: {x_train.max()}")

after scaling  -> Min: 0.0,Max: 1.0


In [11]:
k_values={1,3,5,7,9,15}
results={}
print(f"{'k Values':<8} |{'Accuarcy':<10} | {'prediction time (seconds)':<25}")
print("_"*50)
for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean',n_jobs=-1)
    knn.fit(x_train,y_train)
    start_time=time.time()
    y_pred=knn.predict(x_test)
    elapsed_time=time.time()-start_time
    acc=accuracy_score(y_test,y_pred)
    results[k]={
    "accuracy":acc,
    "time":elapsed_time,
    "predictions":y_pred
    }
    print(f"{k:<8} | {acc* 100:<9.2f}% | {elapsed_time:<25.2f}")

k Values |Accuarcy   | prediction time (seconds)
__________________________________________________
1        | 79.30    % | 2.66                     
3        | 80.90    % | 0.13                     
5        | 81.00    % | 0.13                     
7        | 81.10    % | 0.13                     
9        | 80.95    % | 0.13                     
15       | 80.25    % | 0.13                     


In [12]:
class_names={
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "sandal","shirt","sneaker","bag","ankle boot"
}
best_k=max(results,key=lambda k: results[k]["accuracy"])
print(f"Best K is : {best_k} with {results[best_k]['accuracy']*100:.2f} %accuracy\n")

print("pers-Class Classification Report :")
print(classification_report(y_test,results[best_k]["predictions"],target_names=class_names))

Best K is : 7 with 81.10 %accuracy

pers-Class Classification Report :
              precision    recall  f1-score   support

       Dress       0.73      0.83      0.78       200
     sneaker       0.97      0.94      0.96       200
       shirt       0.67      0.73      0.70       200
        Coat       0.85      0.83      0.84       200
      sandal       0.74      0.69      0.72       200
    Pullover       1.00      0.76      0.86       200
     Trouser       0.58      0.55      0.56       200
  ankle boot       0.81      0.92      0.86       200
         bag       0.97      0.94      0.95       200
 T-shirt/top       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

